In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.decomposition import PCA
from bilinear_model import *
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [ ]:
mean_df = pd.read_csv("connection_matrix_mean.csv", index_col = 0)
var_df = pd.read_csv("connection_matrix_var.csv", index_col = 0)

In [ ]:
mean_df = mean_df.rename(index={'xbc': 'bc5xbc'})
mean_df = mean_df.sort_index().sort_index(axis=1)
var_df = var_df.rename(index={'xbc': 'bc5xbc'})
var_df = var_df.sort_index().sort_index(axis=1)

In [ ]:
sns.heatmap(mean_df, cmap='coolwarm')

In [ ]:
sns.heatmap(var_df, cmap='coolwarm')

In [ ]:
with open('rgc_df_gene_selected.pkl', 'rb') as f:
    rgc_df = pickle.load(f)
rgc_df_all = rgc_df.copy()
rgc_df_all

In [ ]:
df = pd.read_csv("RGC_type_correspondence_combined.csv")
df

In [ ]:
# Initialize an empty dictionary
rgc_map = {}

# Iterate over the rows of the dataframe
for index, row in df.iterrows():
    # Extract the value and key from the row
    value = row['Bae et al. (2018) Eyewire type']
    key = row['Tran et al. (2019) Cluster']
    
    # Extract the number from the 'key' column
    key_number = int(key[1:])
    
    # Adjust the key number by subtracting 1
    adjusted_key = key_number - 1
    
    # Add the key-value pair to the dictionary
    rgc_map[adjusted_key] = value

# Print the resulting dictionary
print(rgc_map)

In [ ]:
# Annotate data
rgc_df = rgc_df[rgc_df.index.isin(rgc_map.keys())]
rgc_df = rgc_df.rename(index = rgc_map)
rgc_df

In [ ]:
# Standardization
scaler = StandardScaler()
rgc_df_values = scaler.fit_transform(rgc_df.values)
rgc_df_values

In [ ]:
# Perform PCA
pca = PCA(n_components=rgc_df.shape[1])
pca.fit(rgc_df_values)

# Calculate cumulative explained variance ratios
variance_ratios = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratios)

# Find the number of components that explain at least 95% variance
num_components = np.argmax(cumulative_variance >= 0.95)
explained_variance = cumulative_variance[num_components]

print(f"Number of components explaining at least 95% variance: {num_components+1}")
print(f"Explained variance: {explained_variance}")

# Transform data into PC space using the selected number of components
rgc_df_pca = pd.DataFrame(pca.transform(rgc_df_values)[:, :num_components+1], index=rgc_df.index)

# Print
rgc_pca_components = pca.components_[:num_components+1].T
rgc_df_pca

In [ ]:
with open('bc_df_gene_selected.pkl', 'rb') as f:
    bc_df = pickle.load(f)
bc_df_all = bc_df.copy()
bc_df_all

In [ ]:
bc_map = {
    0:"bc1",
    1:"bc1",
    2:"bc2",
    3:"bc3a",
    4:"bc3b",
    5:"bc4",
    6:"bc5i",
    7:"bc5o",
    8:"bc5t",
    #9:"xbc",
    9:"bc5xbc",
    10:"bc6",
    11:"bc7",
    12:"bc8/9",
    13:"bc8/9",
}

In [ ]:
# Annotate data
bc_df = bc_df[bc_df.index.isin(bc_map.keys())]
bc_df = bc_df.rename(index = bc_map)
bc_df

In [ ]:
# Standardization
scaler = StandardScaler()
bc_df_values = scaler.fit_transform(bc_df.values)
bc_df_values

In [ ]:
# Perform PCA
pca = PCA(n_components=bc_df.shape[0])
pca.fit(bc_df_values)

# Calculate cumulative explained variance ratios
variance_ratios = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratios)

# Find the number of components that explain at least 95% variance
num_components = np.argmax(cumulative_variance >= 0.95)
explained_variance = cumulative_variance[num_components]

print(f"Number of components explaining at least 95% variance: {num_components+1}")
print(f"Explained variance: {explained_variance}")

# Transform data into PC space using the selected number of components
bc_df_pca = pd.DataFrame(pca.transform(bc_df_values)[:, :num_components+1], index=bc_df.index)

# Print
bc_pca_components = pca.components_[:num_components+1].T
bc_df_pca

In [ ]:
rgc_mean_df = rgc_df_pca.groupby(rgc_df_pca.index).mean()
rgc_mean_df = rgc_mean_df.sort_index().sort_index(axis=1)
rgc_var = rgc_df_pca.groupby(rgc_df_pca.index).var().sum()
rgc_mean_df = rgc_mean_df/np.sqrt(rgc_var)
rgc_mean_df

In [ ]:
bc_mean_df = bc_df_pca.groupby(bc_df_pca.index).mean()
bc_mean_df = bc_mean_df.sort_index().sort_index(axis=1)
bc_var = bc_df_pca.groupby(bc_df_pca.index).var().sum()
bc_mean_df = bc_mean_df/np.sqrt(bc_var)
bc_mean_df

In [ ]:
lambdas = [100, 10, 1, 0.1]
ds = [16, 8, 4, 2]
k = 5
masks = create_k_folds(mean_df.values, k)
test_losses = []
for d in ds:
    for l in lambdas:
        r = 0.000001/d
        _, test_loss = cross_validate(Z = mean_df.values, X = bc_mean_df.values, Y = rgc_mean_df.values, masks = masks, n_factors = d, n_iterations=1000000, learning_rate=r, lambda_a=l, lambda_b=l, verbose=False)
        result = [test_loss, d, l, r]
        test_losses.append(result)
        print (result)
        
        filename = f'bilinear_lambda_{l}_dimension_{d}.npy'
        np.save(filename, np.array(result))

In [ ]:
lambdas = [0.1, 1, 10, 100]
ds = [1]
k = 5
masks = create_k_folds(mean_df.values, k)
test_losses = []
for d in ds:
    for l in lambdas:
        r = 0.000001/d
        _, test_loss = cross_validate(Z = mean_df.values, X = bc_mean_df.values, Y = rgc_mean_df.values, masks = masks, n_factors = d, n_iterations=1000000, learning_rate=r, lambda_a=l, lambda_b=l, verbose=False)
        result = [test_loss, d, l, r]
        test_losses.append(result)
        print (result)
        
        filename = f'bilinear_lambda_{l}_dimension_{d}.npy'
        np.save(filename, np.array(result))

In [ ]:
d = 1
l = 1
r = 0.000001/d
_, test_loss = cross_validate(Z = mean_df.values, X = bc_mean_df.values, Y = rgc_mean_df.values, masks = masks, n_factors = d, n_iterations=1000000, learning_rate=r, lambda_a=l, lambda_b=l, verbose=False)
result = [test_loss, d, l, r]
print (result)

In [ ]:
filename = f'bilinear_lambda_{l}_dimension_{d}.npy'
np.save(filename, np.array(result))

In [ ]:
lambdas = [0.1, 1, 10, 100]
ds = [3]
k = 5
masks = create_k_folds(mean_df.values, k)
test_losses = []
for d in ds:
    for l in lambdas:
        r = 0.000001/d
        _, test_loss = cross_validate(Z = mean_df.values, X = bc_mean_df.values, Y = rgc_mean_df.values, masks = masks, n_factors = d, n_iterations=1000000, learning_rate=r, lambda_a=l, lambda_b=l, verbose=False)
        result = [test_loss, d, l, r]
        test_losses.append(result)
        print (result)
        
        filename = f'bilinear_lambda_{l}_dimension_{d}.npy'
        np.save(filename, np.array(result))

In [ ]:
# Initialize the arrays for dimensions (d) and lambdas (l)
ds = [16, 8, 4, 3, 2, 1]
lambdas = [100, 10, 1, 0.1]

# Create an empty array to store the test loss values
test_losses = np.zeros((len(ds), len(lambdas)))

# Load the test loss values from the npy files
for i, d in enumerate(ds):
    for j, l in enumerate(lambdas):
        filename = f'bilinear_lambda_{l}_dimension_{d}.npy'
        result = np.load(filename)
        test_loss = result[0]
        test_losses[i, j] = np.log10(test_loss)

# Create a 2D heatmap plot
plt.imshow(test_losses, cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(lambdas)), lambdas)
plt.yticks(range(len(ds)), ds)
plt.xlabel('Lambda')
plt.ylabel('Dimension')
plt.title ('Log10 of Validation Loss')
plt.show()

In [ ]:
# Find the minimal element and its index
min_value = np.min(test_losses)
min_index = np.unravel_index(np.argmin(test_losses), test_losses.shape)

# Get the values of d and l for the minimal test loss
opt_d = ds[min_index[0]]
opt_l = lambdas[min_index[1]]

# Print the minimal test loss and its corresponding d and l values
print("Minimal Log10 of Test Loss:", min_value)
print("d for Minimal Test Loss:", opt_d)
print("l for Minimal Test Loss:", opt_l)